<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/notebooks/pnp_evaluation_10k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HST 10K — 6-DoF PnP evaluation

Use the trained YOLOv8s-Pose model to predict 37 image keypoints on the **1,000 held-out test images**, estimate Hubble's camera-relative pose with our existing `src/pnp.py` solver, and compare it with the pose derived from the dataset transforms.

Same geometric method as the 1K smoke test: **all 37 predicted points, `SOLVEPNP_ITERATIVE`, no RANSAC or confidence-based keypoint filtering**. Training and YOLO mAP evaluation are not repeated here.

## 1. Setup

In [1]:
!pip -q install ultralytics opencv-python-headless pandas

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import shutil
import subprocess
import sys
from zipfile import ZipFile

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from ultralytics import YOLO

repo = Path('/content/spacecraft-pose-estimation')
if not (repo / 'src/pnp.py').is_file():
    subprocess.run(
        ['git', 'clone', '--quiet', 'https://github.com/EliasNoorzad/spacecraft-pose-estimation.git', str(repo)],
        check=True,
    )
sys.path.insert(0, str(repo / 'src'))
from pnp import solve_pnp

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 2. Load the test images and dataset annotations

In [2]:
HST_DRIVE = Path('/content/drive/MyDrive/HST')
RAW_DATASET = HST_DRIVE / 'DLVS3-HST'
DATASET = Path('/content/hst_yolo_pose_10k')
DATASET_ZIP = HST_DRIVE / 'hst_yolo_pose_10k.zip'
BEST_PT = HST_DRIVE / 'HST_YOLO_Runs/hst_pose_10k_sigma_010/weights/best.pt'
RESULTS_DIR = HST_DRIVE / 'HST_YOLO_Runs/hst_pose_10k_sigma_010'

# Use the extracted dataset if it is already in this Colab session.
if not (DATASET / 'images/test').is_dir():
    if DATASET.exists():
        raise RuntimeError('An incomplete dataset folder is present in /content. Use a fresh runtime.')
    local_zip = Path('/content/hst_yolo_pose_10k.zip')
    shutil.copyfile(DATASET_ZIP, local_zip)
    with ZipFile(local_zip, 'r') as archive:
        archive.extractall('/content')
    local_zip.unlink()

TEST_IMAGES = DATASET / 'images/test'
assert TEST_IMAGES.is_dir(), f'Test images not found: {TEST_IMAGES}'
assert RAW_DATASET.is_dir(), f'Original metadata not found: {RAW_DATASET}'
assert BEST_PT.is_file(), f'Checkpoint not found: {BEST_PT}'
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab.'

with (RAW_DATASET / '_CameraIntrinsicMatrix.json').open(encoding='utf-8') as f:
    K = np.asarray(json.load(f), dtype=np.float64)
with (RAW_DATASET / '_HST_Keypoints.json').open(encoding='utf-8') as f:
    points_3d = np.asarray([[p['X'], p['Y'], p['Z']] for p in json.load(f)], dtype=np.float64)
assert K.shape == (3, 3) and points_3d.shape == (37, 3)

print('GPU:', torch.cuda.get_device_name(0))
print('Test images:', TEST_IMAGES)
print('Checkpoint:', BEST_PT)

GPU: NVIDIA L4
Test images: /content/hst_yolo_pose_10k/images/test
Checkpoint: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010/weights/best.pt


## 3. Match test images to their ground-truth frames

The 10K dataset comes from ten metadata files. During preprocessing, any duplicate image names were renamed and recorded in `filename_map.json`. Use that mapping when present, otherwise match by the unique original filename.

In [3]:
from collections import defaultdict

def frames_from(obj):
    # Same frame traversal and indexing as src/prepare_hst_yolo.py.
    if isinstance(obj, dict):
        if 'ImagePath' in obj or 'TargetKeypoints' in obj:
            yield obj
        else:
            for value in obj.values():
                yield from frames_from(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from frames_from(value)

frames_by_metadata = {}
original_stems = defaultdict(list)
metadata_files = sorted(RAW_DATASET.glob('*_metadata.json'))

for metadata_file in metadata_files:
    with metadata_file.open(encoding='utf-8-sig') as f:
        frames = list(frames_from(json.load(f)))
    frames_by_metadata[metadata_file.name] = frames
    for index, frame in enumerate(frames):
        original_stem = Path(str(frame['ImagePath']).replace('\\', '/')).stem
        original_stems[original_stem].append((metadata_file.name, index))

map_file = DATASET / 'filename_map.json'
filename_map = json.loads(map_file.read_text(encoding='utf-8')) if map_file.is_file() else {}

def find_frame(image_stem):
    if image_stem in filename_map:
        entry = filename_map[image_stem]
        metadata_name, index = entry['metadata'], entry['frame_index']
    else:
        matches = original_stems.get(image_stem, [])
        if len(matches) != 1:
            raise ValueError(f'Cannot uniquely match test image {image_stem!r}: {matches}')
        metadata_name, index = matches[0]
    return frames_by_metadata[metadata_name][index]

test_paths = sorted(TEST_IMAGES.glob('*.png'))
frames_by_test_image = {image.name: find_frame(image.stem) for image in test_paths}

print('Metadata files:', len(metadata_files))
print('Test images matched to metadata:', len(frames_by_test_image))
print('Renamed source filenames recorded:', len(filename_map))

Metadata files: 10
Test images matched to metadata: 1000
Renamed source filenames recorded: 0


## 4. Ground-truth pose and coordinate convention

In [4]:
def matrix4(values):
    return np.asarray(
        [[values[f'M{i}{j}'] for j in range(1, 5)] for i in range(1, 5)],
        dtype=np.float64,
    )

def ground_truth_pose(frame):
    # DLVS3-HST row-vector convention, converted to OpenCV camera coordinates.
    camera = matrix4(frame['CameraTransform'])
    target = matrix4(frame['TargetTransform'])
    relative = target @ camera
    flip_axes = np.diag([1.0, -1.0, -1.0])
    return flip_axes @ relative[:3, :3].T, flip_axes @ relative[3, :3]

def rotation_error(R_pred, R_true):
    cosine = (np.trace(R_pred @ R_true.T) - 1) / 2
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))

# Check that the matched metadata and dataset coordinate conventions agree.
reprojection_errors = []
for image_name, frame in frames_by_test_image.items():
    R_gt, t_gt = ground_truth_pose(frame)
    rvec_gt, _ = cv2.Rodrigues(R_gt)
    projected, _ = cv2.projectPoints(points_3d, rvec_gt, t_gt, K, None)
    projected = projected.reshape(37, 2)
    annotated = np.asarray([[p['X'], p['Y']] for p in frame['TargetKeypoints']], dtype=np.float64)
    valid = np.isfinite(annotated).all(axis=1) & np.isfinite(projected).all(axis=1)
    valid &= (annotated[:, 0] >= 0) & (annotated[:, 0] < 1024)
    valid &= (annotated[:, 1] >= 0) & (annotated[:, 1] < 1024)
    reprojection_errors.extend(np.linalg.norm(projected[valid] - annotated[valid], axis=1))

if not reprojection_errors:
    raise RuntimeError('No valid keypoints were found for the reprojection check.')
median_reprojection_error = float(np.median(reprojection_errors))
print(f'Ground-truth reprojection median: {median_reprojection_error:.4f} px')
if median_reprojection_error > 2:
    raise ValueError('Reprojection mismatch: check the metadata mapping or coordinate transforms.')

Ground-truth reprojection median: 0.0009 px


## 5. Run YOLO + PnP on the test set

In [5]:
model = YOLO(str(BEST_PT))

predictions = model.predict(
    source=str(TEST_IMAGES),
    imgsz=640,
    batch=8,
    conf=0.25,
    device=0,
    save=False,
    verbose=False,
    stream=True,
)

pose_records = []
failed_detection = 0
failed_pnp = 0
processed = 0

for result in predictions:
    processed += 1
    image_name = Path(result.path).name
    frame = frames_by_test_image[image_name]

    if result.keypoints is None or len(result.keypoints.xy) == 0:
        failed_detection += 1
        continue

    # Keep all 37 predicted keypoints, including low-confidence ones.
    points_2d = result.keypoints.xy[0].cpu().numpy()
    if points_2d.shape != (37, 2) or not np.isfinite(points_2d).all():
        failed_pnp += 1
        continue

    try:
        estimate = solve_pnp(points_2d, points_3d, K)
    except cv2.error:
        estimate = None
    if estimate is None:
        failed_pnp += 1
        continue

    R_pred, t_pred = estimate
    R_gt, t_gt = ground_truth_pose(frame)
    pose_records.append({
        'Image': image_name,
        'Translation error (m)': float(np.linalg.norm(t_pred - t_gt)),
        'Rotation error (deg)': rotation_error(R_pred, R_gt),
    })

pose_df = pd.DataFrame(pose_records)
if pose_df.empty:
    raise RuntimeError('No PnP poses were estimated.')

print(f'Test images processed: {processed}')
print(f'Missing YOLO detections: {failed_detection}')
print(f'PnP failures: {failed_pnp}')
print(f'PnP successes: {len(pose_df)}/{processed}')

Test images processed: 1000
Missing YOLO detections: 0
PnP failures: 0
PnP successes: 1000/1000


## 6. Pose estimation errors

In [6]:
def summarize(df):
    translation = df['Translation error (m)']
    rotation = df['Rotation error (deg)']
    return pd.DataFrame({
        'Metric': ['Mean', 'Median', '90th percentile', 'Maximum'],
        'Translation (m)': [translation.mean(), translation.median(), translation.quantile(0.9), translation.max()],
        'Rotation (degrees)': [rotation.mean(), rotation.median(), rotation.quantile(0.9), rotation.max()],
    })

print('========== YOLO-PREDICTED KEYPOINTS + PnP ==========')
display(summarize(pose_df).round(4))

print('========== 10 LARGEST TRANSLATION ERRORS ==========')
display(pose_df.nlargest(10, 'Translation error (m)').round(4))

print('========== 10 LARGEST ROTATION ERRORS ==========')
display(pose_df.nlargest(10, 'Rotation error (deg)').round(4))

pose_output = RESULTS_DIR / 'pnp_test_10k.csv'
pose_df.to_csv(pose_output, index=False)
print('Saved:', pose_output)

========== YOLO-PREDICTED KEYPOINTS + PnP ==========


,Metric,Translation (m),Rotation (degrees)
0,Mean,0.6358,4.6462
1,Median,0.2326,2.3508
2,90th percentile,0.6146,5.5193
3,Maximum,253.8642,179.5848


========== 10 LARGEST TRANSLATION ERRORS ==========


,Image,Translation error (m),Rotation error (deg)
604,output_0006065.png,253.8642,150.9007
801,output_0007973.png,12.9509,13.9855
408,output_0004092.png,11.7881,178.2176
78,output_0000887.png,9.6466,107.2649
326,output_0003457.png,8.7878,17.7211
521,output_0005143.png,8.2328,70.1316
355,output_0003683.png,5.7485,178.6258
192,output_0001924.png,5.2379,167.8296
688,output_0006971.png,5.2050,17.6330
251,output_0002579.png,5.0899,171.6299


========== 10 LARGEST ROTATION ERRORS ==========


,Image,Translation error (m),Rotation error (deg)
899,output_0009017.png,0.6890,179.5848
90,output_0000993.png,0.2288,178.6397
355,output_0003683.png,5.7485,178.6258
408,output_0004092.png,11.7881,178.2176
182,output_0001832.png,2.3719,176.8288
251,output_0002579.png,5.0899,171.6299
85,output_0000958.png,1.0798,168.6965
192,output_0001924.png,5.2379,167.8296
604,output_0006065.png,253.8642,150.9007
78,output_0000887.png,9.6466,107.2649


Saved: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010/pnp_test_10k.csv


## 7. PnP with ground-truth 2D keypoints (geometry check)

For the same images where YOLO + PnP succeeded, run the unchanged PnP solver again using the annotated 2D keypoints instead of YOLO predictions. This checks the geometry; it is **not** an independent test of YOLO performance.

In [7]:
gt_records = []
gt_skipped = 0

for image_name in pose_df['Image']:
    frame = frames_by_test_image[image_name]
    gt_2d = np.asarray([[p['X'], p['Y']] for p in frame['TargetKeypoints']], dtype=np.float64)
    if gt_2d.shape != (37, 2) or not np.isfinite(gt_2d).all():
        gt_skipped += 1
        continue

    try:
        estimate = solve_pnp(gt_2d, points_3d, K)
    except cv2.error:
        estimate = None
    if estimate is None:
        gt_skipped += 1
        continue

    R_pred, t_pred = estimate
    R_gt, t_gt = ground_truth_pose(frame)
    gt_records.append({
        'Image': image_name,
        'Translation error (m)': float(np.linalg.norm(t_pred - t_gt)),
        'Rotation error (deg)': rotation_error(R_pred, R_gt),
    })

gt_df = pd.DataFrame(gt_records)
if gt_df.empty:
    raise RuntimeError('Ground-truth keypoint PnP did not produce any poses.')

print(f'Ground-truth keypoint PnP: {len(gt_df)}/{len(pose_df)} images')
print(f'Skipped or failed: {gt_skipped}')
display(summarize(gt_df).round(6))

gt_output = RESULTS_DIR / 'pnp_gt_keypoints_10k.csv'
gt_df.to_csv(gt_output, index=False)
print('Saved:', gt_output)

Ground-truth keypoint PnP: 1000/1000 images
Skipped or failed: 0


,Metric,Translation (m),Rotation (degrees)
0,Mean,0.000023,0.009438
1,Median,0.000020,0.000000
2,90th percentile,0.000043,0.026846
3,Maximum,0.000068,0.040213


Saved: /content/drive/MyDrive/HST/HST_YOLO_Runs/hst_pose_10k_sigma_010/pnp_gt_keypoints_10k.csv
